In [1]:
import torch
from mmcv_ops.knn import knn_forward_cuda, knn_forward_cpu

### 1. knn_forward_cuda
We first read the source code of `knn_forward_cuda`. Note that center_xyz is new points without labels, and xyz is the labeled data.
```python
knn = KNN.apply


def knn_forward_cuda(k: int,
                     xyz: torch.Tensor,
                     center_xyz: Optional[torch.Tensor] = None,
                     transposed: bool = False) -> torch.Tensor:
    return knn(k, xyz, center_xyz, transposed)
```

### 2. KNN Class
```python
class KNN(Function):
    r"""KNN (CUDA) based on heap data structure.

    Modified from `PAConv <https://github.com/CVMI-Lab/PAConv/tree/main/
    scene_seg/lib/pointops/src/knnquery_heap>`_.

    Find k-nearest points.
    """

    @staticmethod
    def forward(ctx,
                k: int,
                xyz: torch.Tensor,
                center_xyz: Optional[torch.Tensor] = None,
                transposed: bool = False) -> torch.Tensor:
        """
        Args:
            k (int): number of nearest neighbors.
            xyz (torch.Tensor): (B, N, 3) if transposed == False, else
                (B, 3, N). xyz coordinates of the features.
            center_xyz (torch.Tensor, optional): (B, npoint, 3) if transposed
                is False, else (B, 3, npoint). centers of the knn query.
                Default: None.
            transposed (bool, optional): whether the input tensors are
                transposed. Should not explicitly use this keyword when
                calling knn (=KNN.apply), just add the fourth param.
                Default: False.

        Returns:
            torch.Tensor: (B, k, npoint) tensor with the indices of the
            features that form k-nearest neighbours.
        """
        assert (k > 0) & (k < 100), 'k should be in range(0, 100)'

        if center_xyz is None:
            center_xyz = xyz

        if transposed:
            xyz = xyz.transpose(2, 1).contiguous()
            center_xyz = center_xyz.transpose(2, 1).contiguous()

        assert xyz.is_contiguous()  # [B, N, 3]
        assert center_xyz.is_contiguous()  # [B, npoint, 3]

        center_xyz_device = center_xyz.get_device()
        assert center_xyz_device == xyz.get_device(), \
            'center_xyz and xyz should be put on the same device'
        if xyz.device.type != 'npu':
            if torch.cuda.current_device() != center_xyz_device:
                torch.cuda.set_device(center_xyz_device)

        B, npoint, _ = center_xyz.shape
        N = xyz.shape[1]
        # idx is used to save the index of nearest points
        idx = center_xyz.new_zeros((B, npoint, k)).int()
        # dist2 is used to save the distance
        dist2 = center_xyz.new_zeros((B, npoint, k)).float()

        ext_module.knn_forward_cuda(xyz,
                                    center_xyz,
                                    idx,
                                    dist2,
                                    b=B,
                                    n=N,
                                    m=npoint,
                                    nsample=k)
        # idx shape to [B, k, npoint]
        idx = idx.transpose(2, 1).contiguous()
        if torch.__version__ != 'parrots':
            ctx.mark_non_differentiable(idx)
        return idx

    @staticmethod
    def backward(ctx, a=None):
        return None, None, None
```

### 3. knn_forward_cuda
```cpp
void knn_forward_cuda(Tensor xyz, Tensor new_xyz, Tensor idx, Tensor dist2, int b, 
                      int n, int m, int nsample) {
  // param new_xyz: (B, m, 3)
  // param xyz: (B, n, 3)
  // param idx: (B, m, nsample)

  at::cuda::CUDAGuard device_guard(new_xyz.device());
  cudaStream_t stream = at::cuda::getCurrentCUDAStream();

  // blockIdx.x(col), blockIdx.y(row)
  dim3 blocks(GET_BLOCKS(m, THREADS_PER_BLOCK), b);
  dim3 threads(THREADS_PER_BLOCK);

  AT_DISPATCH_FLOATING_TYPES_AND_HALF(
      new_xyz.scalar_type(), "knn_forward_cuda_kernel", [&] {
        knn_forward_cuda_kernel<scalar_t><<<blocks, threads, 0, stream>>>(
            b, n, m, nsample, xyz.data_ptr<scalar_t>(),
            new_xyz.data_ptr<scalar_t>(), idx.data_ptr<int>(),
            dist2.data_ptr<scalar_t>());
      });

  AT_CUDA_CHECK(cudaGetLastError());
}
```

### 4. Swap Function
```cpp
inline __device__ void swap_float(float *x, float *y) {
  float tmp = *x;
  *x = *y;
  *y = tmp;
}

inline __device__ void swap_int(int *x, int *y) {
  int tmp = *x;
  *x = *y;
  *y = tmp;
}
```

### 5. Heap Sort
```cpp
__device__ void reheap(float *dist, int *idx, int k) {
  int root = 0;
  int child = root * 2 + 1;
  while (child < k) {
    if (child + 1 < k && dist[child + 1] > dist[child]) child++;
    if (dist[root] > dist[child]) return;
    swap_float(&dist[root], &dist[child]);
    swap_int(&idx[root], &idx[child]);
    root = child;
    child = root * 2 + 1;
  }
}

__device__ void heap_sort(float *dist, int *idx, int k) {
  int i;
  for (i = k - 1; i > 0; i--) {
    swap_float(&dist[0], &dist[i]);
    swap_int(&idx[0], &idx[i]);
    reheap(dist, idx, i);
  }
}
```

### 6. knn_forward_cuda_kernel
```cpp
// input: xyz (b, n, 3) new_xyz (b, m, 3)
// output: idx (b, m, nsample) dist2 (b, m, nsample)
template <typename T>
__global__ void knn_forward_cuda_kernel(int b, int n, int m, int nsample,
                                        const T *xyz, const T *new_xyz,
                                        int *__restrict__ idx, T *dist2) {
  // blockIdx.y is the batch idx
  int bs_idx = blockIdx.y;
  // grid-stride loop
  CUDA_1D_KERNEL_LOOP(pt_idx, m) {
    if (bs_idx >= b) return;

    new_xyz += bs_idx * m * 3 + pt_idx * 3;
    xyz += bs_idx * n * 3;
    idx += bs_idx * m * nsample + pt_idx * nsample;
    dist2 += bs_idx * m * nsample + pt_idx * nsample;

    T new_x = new_xyz[0];
    T new_y = new_xyz[1];
    T new_z = new_xyz[2];

    float best_dist[100];
    int best_idx[100];
    for (int i = 0; i < nsample; i++) {
      best_dist[i] = 1e10;
      best_idx[i] = 0;
    }
    for (int i = 0; i < n; i++) {
      T x = xyz[i * 3 + 0];
      T y = xyz[i * 3 + 1];
      T z = xyz[i * 3 + 2];
      T d2 = (new_x - x) * (new_x - x) + (new_y - y) * (new_y - y) +
             (new_z - z) * (new_z - z);
      if (d2 < best_dist[0]) {
        best_dist[0] = d2;
        best_idx[0] = i;
        reheap(best_dist, best_idx, nsample);
      }
    }
    heap_sort(best_dist, best_idx, nsample);
    for (int i = 0; i < nsample; i++) {
      idx[i] = best_idx[i];
      dist2[i] = best_dist[i];
    }
  }
}
```

In [2]:
k = 5
batch_size = 1
N = 5
M = 6
xyz = torch.randn(batch_size, N, 3).cuda()
center_xyz = torch.randn(batch_size, M, 3).cuda()

In [3]:
print(knn_forward_cuda(k, xyz, center_xyz))

tensor([[[3, 3, 1, 3, 0, 0],
         [2, 1, 4, 0, 2, 3],
         [0, 2, 0, 1, 3, 2],
         [1, 0, 3, 2, 1, 1],
         [4, 4, 2, 4, 4, 4]]], device='cuda:0', dtype=torch.int32)


### 7. knn_forward_cpu_kernel
I write a cpu kernel for knn, it is almost the same with cuda kernel.
```cpp
template <typename T>
void knn_forward_cpu_kernel(int b, int n, int m, int nsample,
                                        const T *xyz, const T *new_xyz,
                                        int *idx, T *dist2) {
  for(int bs_idx = 0; bs_idx < b; bs_idx++){
    for(int pt_idx = 0; pt_idx < m; pt_idx++){
        const T* cur_new_xyz = new_xyz + bs_idx * m * 3 + pt_idx * 3;
        const T* cur_xyz = xyz + bs_idx * n * 3;
        int* cur_idx = idx + bs_idx * m * nsample + pt_idx * nsample;
        T* cur_dist2 = dist2 + bs_idx * m * nsample + pt_idx * nsample;
        T new_x = cur_new_xyz[0];
        T new_y = cur_new_xyz[1];
        T new_z = cur_new_xyz[2];
        float best_dist[100];
        int best_idx[100];
        for (int i = 0; i < nsample; i++) {
            best_dist[i] = 1e10;
            best_idx[i] = 0;
        }

        for (int i = 0; i < n; i++) {
            T x = cur_xyz[i * 3 + 0];
            T y = cur_xyz[i * 3 + 1];
            T z = cur_xyz[i * 3 + 2];
            T d2 = (new_x - x) * (new_x - x) + (new_y - y) * (new_y - y) +
                    (new_z - z) * (new_z - z);
            if (d2 < best_dist[0]) {
                best_dist[0] = d2;
                best_idx[0] = i;
                reheap_cpu(best_dist, best_idx, nsample);
            }
        }
        heap_sort_cpu(best_dist, best_idx, nsample);
        for (int i = 0; i < nsample; i++) {
            cur_idx[i] = best_idx[i];
            cur_dist2[i] = best_dist[i];
        }
    }
  }
}
```

In [4]:
print(knn_forward_cpu(k, xyz.cpu(), center_xyz.cpu()))

tensor([[[3, 3, 1, 3, 0, 0],
         [2, 1, 4, 0, 2, 3],
         [0, 2, 0, 1, 3, 2],
         [1, 0, 3, 2, 1, 1],
         [4, 4, 2, 4, 4, 4]]], dtype=torch.int32)
